In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import yt
yt.set_log_level('error')
import os
from dotenv import dotenv_values
import glob
import pandas as pd
import re
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.stats import linregress
from scipy.stats import norm
import matplotlib.gridspec as gridspec

In [ ]:
kmax_start=3
kmax_end=1000
kmax_inc=25
kmax_arr = np.arange(kmax_start, kmax_end + 1, kmax_inc)
print(f"len(kmax_arr): {len(kmax_arr)}")

chk_arr = np.array([32000])
# chk_arr = np.array([42000, 42500, 43000])
# chk_arr = np.array([90000, 140000, 180000])
# chk_arr = np.array([180000, 260000])

# Nx, Ny = 512, 512
Nx, Ny = 2048, 2048

In [ ]:
def read_in_bar_data(run_dir, kmax, chk_j):
    f_dir = os.path.join(run_dir, f"filtered_{chk_j}_0_{kmax}")
    files = sorted(glob.glob(f_dir))
    # print(files)
    ts = yt.DatasetSeries(files)
    return ts

# def round_to_6_digits(val):
#     # Calculate how many integer digits there are
#     int_len = len(str(abs(int(val))))
#     # Round to the remaining number of decimal places
#     return round(val, 6 - int_len)
    
# def read_in_prime_data(run_dir, kmax):
#     f_dir = os.path.join(run_dir, f"filtered_9000_{round_to_6_digits(np.sqrt(kmax**2 + .5))}_{100*kmax}")
#     print(f_dir)
#     files = sorted(glob.glob(f_dir))
#     print(files)
#     ts = yt.DatasetSeries(files)
#     return ts
    
f_bar_list = []
# f_prime_list = []
N_seeds = 1
for i in range(N_seeds):
    f_bar_i_arr = []
    # f_prime_i_arr = []
    for chk_j in chk_arr:
        f_bar_i_arr_j = []
        # run_dir_i = f"Kolmogorov_WENO_512_1em4_cell_depth_1em5_F_0_1em1_n_WN_1_dt_3em4_seed_{i+1}00"
        run_dir_i = f"Kolmogorov_WENO_2048_mu_1em4_cell_depth_1p6em4_F_0_1em1_n_WN_1_dt_9em5_seed_{i+1}00"
        for kmax in kmax_arr:
            f_bar_i_kmax = read_in_bar_data(run_dir_i, kmax, chk_j)
            # f_prime_i_kmax = read_in_prime_data(run_dir_i, kmax)
            f_bar_i_arr_j.append(f_bar_i_kmax)
        
        f_bar_i_arr.append(f_bar_i_arr_j)
        # f_prime_i_arr.append(f_prime_i_kmax)
        
    f_bar_list.append(f_bar_i_arr)
    # f_prime_list.append(f_prime_i_arr)



In [ ]:
f_bar_list[0][0][0][0].field_list

In [ ]:
delta_nu_arr_chk_seeds = []
for i in range(N_seeds):
    delta_nu_arr_chk = []
    for chk in range(len(chk_arr)):
        # Assemble data:
        delta_nu_arr = []
        for j in range(len(kmax_arr)):
            ds = f_bar_list[i][chk][j][0]
            # cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
            first_grid = ds.index.grids[0]
            # delta_nu = np.array(cg['delta_nu'][0,0,0].v)
            # delta_nu_val = float(cg[('boxlib', 'delta_nu')][0, 0, 0].v)
            delta_nu_val = float(first_grid[('boxlib', 'delta_nu')].flat[0])
            delta_nu_arr.append(delta_nu_val)
        delta_nu_arr = np.array(delta_nu_arr)
        # delta_nu_arr = delta_nu_arr / np.max(delta_nu_arr)
        print(f"chk: {chk_arr[chk]}, delta_nu_arr: {delta_nu_arr}", flush = True)
        delta_nu_arr_chk.append(delta_nu_arr)
    delta_nu_arr_chk_seeds.append(delta_nu_arr_chk)

In [ ]:
for i in range(N_seeds):
    for chk in range(len(chk_arr)):
        delta_nu_arr = delta_nu_arr_chk_seeds[i][chk]
        # delta_nu_arr = delta_nu_arr / np.max(delta_nu_arr)
        # print(f"chk: {chk_arr[chk]}, delta_nu_arr: {delta_nu_arr}", flush = True)
        kmax_np = np.array(kmax_arr)
        L_arr = 1.0 / kmax_np
        plt.scatter(L_arr, delta_nu_arr, label=f'Seed {i+1}00, chk: {chk_arr[chk]}')

plt.xlabel("L")
plt.ylabel('delta_nu')

plt.legend()
plt.show()
plt.close()

In [ ]:
plt.figure(dpi=300)
all_L = []
all_delta_nu = []

for i in range(N_seeds):
    for chk in range(len(chk_arr)):
        delta_nu_arr = delta_nu_arr_chk_seeds[i][chk]
        kmax_np = np.array(kmax_arr)
        # L_arr = 1.0 / kmax_np
        L_arr = 2 * np.pi / kmax_np
        
        plt.scatter(L_arr, delta_nu_arr, label=f'Seed {i+1}00, chk: {chk_arr[chk]}')
        
        # Accumulate the data for the global fit
        all_L.extend(L_arr)
        all_delta_nu.extend(delta_nu_arr)

# Convert aggregated lists into NumPy arrays
all_L = np.array(all_L)
all_delta_nu = np.array(all_delta_nu)

# Perform the global fit y = A * ln(x) + B
A, B = np.polyfit(np.log(all_L), all_delta_nu, 1)

# Generate a dense set of L values to draw a smooth theoretical curve
L_smooth = np.linspace(np.min(all_L), np.max(all_L), 200)
fit_curve = A * np.log(L_smooth) + B

# Plot the smooth fit curve
# Using .2e prints in scientific notation (e.g., 1.23e-04)
# The + flag for B ensures it correctly formats as + 1.23e-04 or - 1.23e-04
plt.plot(L_smooth, fit_curve, color='black', linestyle='--', linewidth=2, 
         label=f'Global Fit: {A:.2e}*ln(L) {B:+.2e}')

plt.xlabel("L")
plt.ylabel(r'$\Delta \nu$')

plt.legend()
plt.show()
plt.close()

In [ ]:
plt.figure(dpi=300)

all_L = []
all_delta_nu = []

for i in range(N_seeds):
    for chk in range(len(chk_arr)):
        delta_nu_arr = delta_nu_arr_chk_seeds[i][chk]
        kmax_np = np.array(kmax_arr)
        
        L_arr = (2.0 * np.pi) / kmax_np
        
        plt.scatter(L_arr, delta_nu_arr, label=f'Seed {i+1}00, chk: {chk_arr[chk]}')
        
        all_L.extend(L_arr)
        all_delta_nu.extend(delta_nu_arr)

all_L = np.array(all_L)
all_delta_nu = np.array(all_delta_nu)

A, B = np.polyfit(np.log(all_L), all_delta_nu, 1)

# Use geomspace for evenly spaced points on a log axis
L_smooth = np.geomspace(np.min(all_L), np.max(all_L), 200)
fit_curve = A * np.log(L_smooth) + B

plt.plot(L_smooth, fit_curve, color='black', linestyle='--', linewidth=2, 
         label=f'Global Fit: {A:.2e}*ln(L) {B:+.2e}')

# Set the x-axis to a logarithmic scale
plt.xscale('log')

plt.xlabel("Length scale L (2π/k_max)")
plt.ylabel(r'$\Delta \nu$')

# Adding a grid helps visually anchor the straight line on log plots
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.legend()
plt.show()
plt.close()

In [ ]:

def plot_delta_nu_fits(N_seeds, chk_arr, kmax_arr, delta_nu_arr_chk_seeds, dpi=300, x_log = False):
    """
    Plots delta_nu vs L for multiple seeds and performs a logarithmic 
    fit (y = A * ln(x) + B) for each checkpoint (chk) value.
    """
    plt.figure(dpi=dpi)

    # Extract the default matplotlib color cycle
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # Loop over chk first to group all seeds for a specific chk together
    for chk_idx in range(len(chk_arr)):
        chk_val = chk_arr[chk_idx]
        
        # Assign a specific color to this chk value, cycling if there are many
        c = colors[chk_idx % len(colors)]
        
        chk_L = []
        chk_delta_nu = []
        
        # Loop over all seeds to gather data for this specific chk
        for i in range(N_seeds):
            delta_nu_arr = delta_nu_arr_chk_seeds[i][chk_idx]
            kmax_np = np.array(kmax_arr)
            L_arr = 2 * np.pi / kmax_np
            
            # Scatter plot for each seed. 
            # Using alpha=0.4 makes the dots lighter.
            # We only assign the scatter label on the first seed to keep the legend clean.
            scatter_label = f'Data (All seeds), chk: {chk_val}' if i == 0 else None
            
            plt.scatter(L_arr, delta_nu_arr, color=c, alpha=0.4, label=scatter_label)
            
            # Accumulate the data for this chk's fit
            chk_L.extend(L_arr)
            chk_delta_nu.extend(delta_nu_arr)

        # Convert aggregated lists into NumPy arrays for the fit
        chk_L = np.array(chk_L)
        chk_delta_nu = np.array(chk_delta_nu)

        # Perform the fit for this specific chk: y = A * ln(x) + B
        A, B = np.polyfit(np.log(chk_L), chk_delta_nu, 1)

        # Generate a dense set of L values to draw a smooth theoretical curve
        L_smooth = np.linspace(np.min(chk_L), np.max(chk_L), 200)
        fit_curve = A * np.log(L_smooth) + B

        # Plot the smooth fit curve fully opaque so it appears darker/bolder
        plt.plot(L_smooth, fit_curve, color=c, linestyle='--', linewidth=2, 
                 label=f'Fit chk {chk_val}: {A:.2e}*ln(L) {B:+.2e}')

    plt.xlabel("L")
    plt.ylabel(r'$\Delta \nu$')

    if x_log:
        plt.xscale('log')

    # Move the legend slightly outside the plot area so it doesn't overlap the curves
    # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.legend()
    plt.tight_layout()

    plt.show()
    plt.close()


plot_delta_nu_fits(N_seeds, chk_arr, kmax_arr, delta_nu_arr_chk_seeds, x_log = False)

plot_delta_nu_fits(N_seeds, chk_arr, kmax_arr, delta_nu_arr_chk_seeds, x_log = True)

In [ ]:
# def plot_field(field, label = "", title = ""):
#     plt.figure(figsize=(8, 6), dpi=150)
#     extent = [0, 1, 0, 1]

#     im = plt.imshow(field, origin='lower', extent=extent, cmap='magma')
#     # im = plt.imshow(trace_2d.T, origin='lower', extent=extent, cmap='magma', 
#     #                 norm=LogNorm(vmin=vmin, vmax=vmax))
    
#     plt.colorbar(im, label=label)

#     # time = ds_final.current_time.v
    
#     plt.title(title)
    
#     plt.xlabel('$x$')
#     plt.ylabel('$y$')
#     plt.show()

In [ ]:
# def outer_v(vel):
#     u = vel[0, :, :]
#     v = vel[1, :, :]

#     uu = u * u
#     uv = u * v
#     vv = v * v

#     return np.array([[uu, uv], [uv, vv]])

# def turb_stress_tensor(uu_bar, vv_bar, uv_bar, vel_bar):
#     vv_tensor_bar = np.array([[uu_bar, uv_bar],
#                               [uv_bar, vv_bar]])
#     # print(f"vel_bar.shape: {vel_bar.shape}", flush = True)
    
#     return vv_tensor_bar - outer_v(vel_bar)


# tau_arr = []
# S_arr = []
# for i in range(N_seeds):
#     tau_arr_seeds = []
#     S_arr_seeds = []
#     for k, kmax in enumerate(kmax_arr):
#         ds = f_bar_list[i][k][0]
#         cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
#         uu_bar = np.array(cg['uu_filter'][:,:,0].v)
#         vv_bar = np.array(cg['vv_filter'][:,:,0].v)
#         uv_bar = np.array(cg['uv_filter'][:,:,0].v)
#         vel_x_bar = np.array(cg['velx_filter'][:,:,0].v)
#         vel_y_bar = np.array(cg['vely_filter'][:,:,0].v)
#         vel_bar = np.array([vel_x_bar, vel_y_bar])
#         tau = turb_stress_tensor(uu_bar, vv_bar, uv_bar, vel_bar)

#         S11 = np.array(cg['S11'][:,:,0].v)
#         S12 = np.array(cg['S12'][:,:,0].v)
#         S22 = np.array(cg['S22'][:,:,0].v)
#         S_t = np.array([[S11, S12], [S12, S22]])
#         if k < 1:
#             plot_field(tau[0,0,:,:], label = r'$\tau$', title = f'Turbulent stress tensor for kmax = {kmax}')
#             plot_field(S_t[0,0,:,:], label = r'$S_{00}$', title = f'Strain rate tensor for kmax = {kmax}')
#         tau_arr_seeds.append(tau)
#         S_arr_seeds.append(S_t)
#     tau_arr.append(tau_arr_seeds)
#     S_arr.append(S_arr_seeds)

# Analyzing $\Delta \eta$

In [ ]:
def read_in_spectrum_data(run_dir, chk, kmax_arr, verbose = False):
    # spect_dir = os.path.join(run_dir, f"Delta_eta_spectrum_0_*{chk}.txt")
    # files = sorted(glob.glob(spect_dir))
    # print(f"len(files): {len(files)}")
    dfs = []
    # for f in files:
    #     num = int(re.search(rf'Delta_eta_spectrum_0_(\d+){chk}\.txt', f).group(1))
    #     df = pd.read_csv(f, sep='\s+', header=None)
    #     df['file_num'] = num
    #     dfs.append(df)

    for k in kmax_arr:
        f = os.path.join(run_dir, f"Delta_eta_spectrum_0_{k}_{chk}.txt")
        df = pd.read_csv(f, sep='\s+', header=None)
        if verbose and k == kmax_arr[0]:
            print(df)
        df['k_cutoff'] = k
        dfs.append(df)
        # if verbose and k == kmax_arr[0]:
        #     print(df)
    spect_data = pd.concat(dfs, ignore_index=True)
    return spect_data
    
eta_spect_data_list = []
N_seeds = 1
for i in range(N_seeds):
    eta_spect_data_chk = []
    for chk in chk_arr:
        run_dir_i = f"Kolmogorov_WENO_2048_mu_1em4_cell_depth_1p6em4_F_0_1em1_n_WN_1_dt_9em5_seed_{i+1}00"
        # if chk == 90000:
        #     eta_spect_data_i = read_in_spectrum_data(run_dir_i, chk, kmax_arr, verbose = True)
        # else:
        eta_spect_data_i = read_in_spectrum_data(run_dir_i, chk, kmax_arr, verbose = False)
        eta_spect_data_chk.append(eta_spect_data_i)
    eta_spect_data_list.append(eta_spect_data_chk)


In [ ]:

def add_power_law_fit(ax, k_vals, E_vals, k_min, k_max, color):
    """Fits E(k) = c * k^p, calculates R^2, and plots the fit."""
    mask = (k_vals >= k_min) & (k_vals <= k_max)
    k_fit, E_fit = k_vals[mask], E_vals[mask]
    
    if len(k_fit) < 2: 
        return
    
    # Linear regression in log-log space
    log_k, log_E = np.log10(k_fit), np.log10(E_fit)
    slope, intercept, r_value, _, _ = linregress(log_k, log_E)
    r_squared = r_value**2
    
    # Plot the fit line
    fit_line = (10**intercept) * (k_fit**slope)
    label = f'Fit: slope={slope:.2f}, $R^2$={r_squared:.2f}'
    ax.plot(k_fit, fit_line, color=color, linestyle='--', linewidth=2, label=label)

def plot_all_loaded_spectra(eta_spect_data_list, N_seeds, chk_arr, 
                            physical_wavenumber=False, bin_width=1, 
                            k_min_g=10, k_max_g=255, k_min_o=2, k_max_o=20,
                            kB=1.380649e-16, T=294.0, rho=1.0, dpi=300):
    """
    Loops through the nested list of spectral data and generates a plot 
    for every seed and checkpoint combination.
    """
    # Loop over all seeds
    for i in range(N_seeds):
        # Loop over all checkpoints
        for chk_idx, chk in enumerate(chk_arr):
            
            # Extract the specific DataFrame for this seed and chk
            spect_data = eta_spect_data_list[i][chk_idx]
            
            # Skip if the dataframe happens to be empty
            # if spect_data is None or spect_data.empty:
            #     print(f"No data found for Seed {i+1}00, chk {chk}. Skipping plot.")
            #     continue
                
            fig, ax = plt.subplots(figsize=(10, 7), dpi=dpi)
            
            # Set up colormap based on k_cutoff
            k_cutoff_arr = sorted(spect_data['k_cutoff'].unique())
            norm = mcolors.Normalize(vmin=min(k_cutoff_arr), vmax=max(k_cutoff_arr))
            cmap = cm.coolwarm

            fit_k, fit_E = None, None

            # Plot each time step for this specific dataframe
            for num, group in spect_data.groupby('k_cutoff'):
                if num < kmax_arr[-1]:
                    continue
                raw_k, raw_E = group[0].values, group[1].values
                
                num_bins = len(raw_k) // bin_width
                k_binned = raw_k[:num_bins*bin_width].reshape(-1, bin_width).mean(axis=1)
                
                if physical_wavenumber:
                    k_binned = 2 * np.pi * k_binned
                    
                E_binned = raw_E[:num_bins*bin_width].reshape(-1, bin_width).mean(axis=1)
                
                ax.plot(k_binned, E_binned, color=cmap(norm(num)))

                ax.axvline(num, color=cmap(norm(num)), linestyle="--", label = f"k = {num}")
                
                # if num == kmax_arr[0]:
                # print(num)
                fit_k, fit_E = k_binned, E_binned

            ax.set_xscale('log')
            ax.set_yscale('log')

            # Add Reference Lines
            FDT = (kB * T) / rho
            if physical_wavenumber:
                # ax.plot(fit_k, FDT * fit_k, 'k--', label=r'Ref: $(k_B T / \rho) k^{1}$') 
                # ax.axvline(x=2 * np.pi, color="red", linestyle="--", label=r"$k=2\pi$")
                ax.set_xlabel(r'$2 \pi k$')
            else:
                # ax.plot(fit_k, 2 * np.pi * FDT * fit_k, 'k--', label=r'Ref: $(2 \pi k_B T / \rho) k^{1}$') 
                # ax.axvline(x=1, color="red", linestyle="--", label=r"$k=1$")
                ax.set_xlabel('k')

            ax.set_ylabel('E(k)')

            # Add power law fits to the final time step
            if physical_wavenumber:
                add_power_law_fit(ax, fit_k, fit_E, k_min=2 * np.pi * k_min_g, k_max=2 * np.pi * k_max_g, color='green')
            else:
                add_power_law_fit(ax, fit_k, fit_E, k_min=k_min_g, k_max=k_max_g, color='green')
                add_power_law_fit(ax, fit_k, fit_E, k_min=k_min_o, k_max=k_max_o, color='orange')

            # Ensure we only extract the legend for the reference lines and fits
            # handles, labels = ax.get_legend_handles_labels()
            # num_legend_items = 3 if physical_wavenumber else 4
            
            # Prevent indexing errors if data resulted in fewer legend items
            # if len(handles) >= num_legend_items:
            #     ax.legend(handles[-num_legend_items:], labels[-num_legend_items:], loc='lower left')
            # else:
            #     ax.legend(loc='lower left')
            ax.legend()

            # Add colorbar
            sm = cm.ScalarMappable(cmap=cmap, norm=norm)
            fig.colorbar(sm, ax=ax, label='k cutoff')
            field_name = r"$\Delta \eta$"
            title = (f"Spectra for {field_name} for Kolmogorov flow using Incflo WENO5 (Bin Width = {bin_width})\n"
                     f"(Nx=512, visc = 1e-4, dt = 1.5e-4, cell_depth = 1e-5, F_0 = .1, n = 1)\n"
                     f"Seed {i+1}00 | Chk {chk}")
            plt.title(title)
            
            plt.tight_layout()
            plt.show()
            plt.close()

# plot_all_loaded_spectra(eta_spect_data_list, N_seeds, chk_arr)
plot_all_loaded_spectra(eta_spect_data_list, N_seeds, chk_arr, physical_wavenumber=False, bin_width=1, k_min_g=3, k_max_g=98, 
                        k_min_o=250, k_max_o=2000, kB=1.380649e-16, T=294.0, rho=1.0, dpi=300)

In [ ]:

def plot_eta_pdf(N_seeds, chk_arr, eta_data_chk_seeds, component_name=r'$\Delta \eta$', dpi=300, y_log=False):
    """
    Plots the Probability Density Function (PDF) of a spatial field for multiple seeds 
    and overlays a Gaussian fit for each checkpoint (chk) value.
    """
    plt.figure(dpi=dpi)

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for chk_idx in range(len(chk_arr)):
        chk_val = chk_arr[chk_idx]
        c = colors[chk_idx % len(colors)]
        
        chk_eta = []
        
        for i in range(N_seeds):
            # Extract the raw 1D array of eta values for this specific seed and checkpoint
            eta_arr = eta_data_chk_seeds[i][chk_idx]
            
            # Accumulate the data to build a robust global histogram for this checkpoint
            chk_eta.extend(eta_arr)

        chk_eta = np.array(chk_eta)

        # Plot the histogram as a probability density (area under the curve equals 1)
        # Using a high number of bins to get a smooth PDF representation
        plt.hist(chk_eta, bins=150, density=True, color=c, alpha=0.4, 
                 label=f'Data (All seeds), chk: {chk_val}')

        # Fit a normal (Gaussian) distribution to the aggregated data
        mu, std = norm.fit(chk_eta)

        # Generate a dense set of x values to draw a smooth theoretical Gaussian curve
        xmin, xmax = plt.xlim()
        x_smooth = np.linspace(xmin, xmax, 300)
        pdf_curve = norm.pdf(x_smooth, mu, std)

        # Plot the smooth Gaussian fit curve fully opaque so it appears darker
        plt.plot(x_smooth, pdf_curve, color=c, linestyle='--', linewidth=2, 
                 label=f'Gaussian Fit: $\mu$={mu:.2e}, $\sigma$={std:.2e}')

    plt.xlabel(component_name)
    plt.ylabel("Probability Density")

    if y_log:
        plt.yscale('log')

    # Move the legend slightly outside the plot area so it doesn't overlap the curves
    # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.legend()
    plt.tight_layout()

    plt.show()
    plt.close()

In [ ]:

# These will hold the data structured as [kmax][seed][chk]
eta_11_all_kmax = []
eta_12_all_kmax = []
eta_22_all_kmax = []

# 1. Loop over kmax FIRST
for j in range(len(kmax_arr)):
    eta_11_arr_seeds = []
    eta_12_arr_seeds = []
    eta_22_arr_seeds = []
    
    # 2. Then loop over seeds
    for i in range(N_seeds):
        eta_11_arr_chk = []
        eta_12_arr_chk = []
        eta_22_arr_chk = []
        
        # 3. Finally loop over checkpoints
        for chk in range(len(chk_arr)):
            # Notice that f_bar_list is still indexed the exact same way: [i][chk][j][0]
            ds = f_bar_list[i][chk][j][0]
            ad = ds.all_data()
            
            # Extract and append the 1D arrays directly to the checkpoint lists
            eta_11_arr_chk.append(np.array(ad[('boxlib', 'delta_eta_11')].v))
            eta_12_arr_chk.append(np.array(ad[('boxlib', 'delta_eta_12')].v))
            eta_22_arr_chk.append(np.array(ad[('boxlib', 'delta_eta_22')].v))
            
        # Append the completed checkpoint lists to the seed lists
        eta_11_arr_seeds.append(eta_11_arr_chk)
        eta_12_arr_seeds.append(eta_12_arr_chk)
        eta_22_arr_seeds.append(eta_22_arr_chk)
        
    print(f"Extracted all seeds and checkpoints for kmax index {j}.", flush=True)
    
    # Append the completed seed lists to the main kmax lists
    eta_11_all_kmax.append(eta_11_arr_seeds)
    eta_12_all_kmax.append(eta_12_arr_seeds)
    eta_22_all_kmax.append(eta_22_arr_seeds)

In [ ]:
# Extract the data for a specific kmax index (e.g., j=0)
kmax_idx = 1

print(f"kmax:{kmax_arr[kmax_idx]}")

plot_eta_pdf(N_seeds, chk_arr, eta_11_all_kmax[kmax_idx], component_name=r'$\Delta \eta_{11}$', y_log=True)

plot_eta_pdf(N_seeds, chk_arr, eta_12_all_kmax[kmax_idx], component_name=r'$\Delta \eta_{12}$', y_log=True)

plot_eta_pdf(N_seeds, chk_arr, eta_22_all_kmax[kmax_idx], component_name=r'$\Delta \eta_{22}$', y_log=True)

In [ ]:
def calculate_eta_covariance(kmax_idx, chk_idx, N_seeds, eta_11, eta_12, eta_22):
    # Use np.concatenate with list comprehensions for much faster array flattening
    eta_11_combined = np.concatenate([eta_11[kmax_idx][i][chk_idx] for i in range(N_seeds)])
    eta_12_combined = np.concatenate([eta_12[kmax_idx][i][chk_idx] for i in range(N_seeds)])
    eta_22_combined = np.concatenate([eta_22[kmax_idx][i][chk_idx] for i in range(N_seeds)])
    
    # Stack the arrays to calculate the covariance matrix
    stacked_eta = np.vstack([eta_11_combined, eta_12_combined, eta_22_combined])
    return np.cov(stacked_eta)

total_var_arr_chk = []


for chk_idx, chk in enumerate(chk_arr):
    total_var_arr = []
    delta_nu_seed_avg_arr = []
    
    # Initialize lists for the new matrix properties
    var_11_arr = []
    var_12_arr = []
    var_22_arr = []
    cov_11_22_arr = []
    cov_11_12_arr = []
    det_cov_arr = []
    
    for kmax_idx, kmax in enumerate(kmax_arr):
        # Calculate covariance matrix
        cov_matrix = calculate_eta_covariance(kmax_idx, chk_idx, N_seeds, 
                                              eta_11_all_kmax, eta_12_all_kmax, eta_22_all_kmax)
        
        # Original calculations
        total_var_arr.append(np.trace(cov_matrix))
        
        seed_values = [delta_nu_arr_chk_seeds[i][chk_idx][kmax_idx] for i in range(N_seeds)]
        delta_nu_seed_avg_arr.append(np.mean(seed_values))
        
        # Extract individual variances (the diagonal elements)
        var_11_arr.append(cov_matrix[0, 0])
        var_12_arr.append(cov_matrix[1, 1])
        var_22_arr.append(cov_matrix[2, 2])
        
        # Extract specific cross-covariances (the off-diagonal elements)
        # 0, 2 is the covariance between 11 and 22 (Incompressibility check)
        cov_11_22_arr.append(cov_matrix[0, 2])
        # 0, 1 is the covariance between 11 and 12 (Isotropy check)
        cov_11_12_arr.append(cov_matrix[0, 1])
        
        # Calculate the determinant of the 3x3 covariance matrix
        det_cov_arr.append(np.linalg.det(cov_matrix))
        
    # Store all the results in the dictionary for this checkpoint
    total_var_arr_chk.append({
        'total_var_arr': total_var_arr, 
        'delta_nu_seed_avg': delta_nu_seed_avg_arr,
        'var_11_arr': var_11_arr,
        'var_12_arr': var_12_arr,
        'var_22_arr': var_22_arr,
        'cov_11_22_arr': cov_11_22_arr,
        'cov_11_12_arr': cov_11_12_arr,
        'det_cov_arr': det_cov_arr
    })

In [ ]:
for chk_idx, chk in enumerate(chk_arr):
    plt.scatter(total_var_arr_chk[chk_idx]['delta_nu_seed_avg'], 
                total_var_arr_chk[chk_idx]['total_var_arr'], 
                label=f"chk = {chk}")

# Use a log-log scale to verify a linear relationship (slope of 1)
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r"$\Delta \nu$")
plt.ylabel("Total variance")
plt.legend()
plt.show()

In [ ]:
%config InlineBackend.figure_format = 'retina'
show_expected = True

# Create a figure with a custom GridSpec layout
fig = plt.figure(figsize=(16, 10), dpi = 600)
gs = gridspec.GridSpec(2, 6, figure=fig)
fig.subplots_adjust(hspace=0.4, wspace=0.8)

# Top row: 3 plots, taking up 2 columns each
ax1 = fig.add_subplot(gs[0, 0:2])
ax2 = fig.add_subplot(gs[0, 2:4])
ax3 = fig.add_subplot(gs[0, 4:6])

# Bottom row: 2 plots, centered by offsetting the columns
ax4 = fig.add_subplot(gs[1, 1:3])
ax5 = fig.add_subplot(gs[1, 3:5])

# Single loop to populate all axes
for chk_idx, chk in enumerate(chk_arr):
    x_data = total_var_arr_chk[chk_idx]['delta_nu_seed_avg']
    
    ax1.scatter(x_data, total_var_arr_chk[chk_idx]['var_11_arr'], label=f"chk = {chk}")
    ax2.scatter(x_data, total_var_arr_chk[chk_idx]['var_22_arr'], label=f"chk = {chk}")
    ax3.scatter(x_data, total_var_arr_chk[chk_idx]['var_12_arr'], label=f"chk = {chk}")
    ax4.scatter(x_data, np.abs(total_var_arr_chk[chk_idx]['cov_11_22_arr']), label=f"chk = {chk}")
    ax5.scatter(x_data, total_var_arr_chk[chk_idx]['cov_11_12_arr'], label=f"chk = {chk}")

# --- Formatting Axes ---

# Ax 1: Variance 11
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r"$\Delta \nu$")
ax1.set_ylabel("Variance of $\Delta\eta_{11}$")
title_1 = "Variance of 11"
if show_expected: title_1 += " (Expect: slope of 1)"
ax1.set_title(title_1)
ax1.legend()

# Ax 2: Variance 22
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel(r"$\Delta \nu$")
ax2.set_ylabel("Variance of $\Delta\eta_{22}$")
title_2 = "Variance of 22"
if show_expected: title_2 += " (Expect: identical to 11)"
ax2.set_title(title_2)
ax2.legend()

# Ax 3: Variance 12
ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_xlabel(r"$\Delta \nu$")
ax3.set_ylabel("Variance of $\Delta\eta_{12}$")
title_3 = "Shear Variance 12"
if show_expected: title_3 += " (Expect: slope of 1)"
ax3.set_title(title_3)
ax3.legend()

# Ax 4: Absolute Covariance 11 and 22
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.set_xlabel(r"$\Delta \nu$")
ax4.set_ylabel("Absolute Covariance (11 & 22)")
title_4 = "Abs Covariance 11 & 22"
if show_expected: title_4 += " (Expect: matches 11 & 22)"
ax4.set_title(title_4)
ax4.legend()

# Ax 5: Covariance 11 and 12
ax5.set_xscale('log')
ax5.set_yscale('symlog', linthresh=1e-15)
ax5.set_xlabel(r"$\Delta \nu$")
ax5.set_ylabel("Covariance (11 & 12)")
title_5 = "Isotropy Check"
if show_expected: title_5 += " (Expect: noise near 0)"
ax5.set_title(title_5)
ax5.legend()

plt.show()



In [ ]:
var_11, var_12, var_22 = np.diag(cov_matrix)


print(f"Total Variance (Trace) : {total_var:.6e}")
print(f"Shear Variance (12)    : {var_12:.6e}")
print(f"Normal Variance (11)/2 : {var_11 / 2:.6e}")
print(f"Normal Variance (22)/2 : {var_22 / 2:.6e}")

In [ ]:
kB = 1.380649e-16
T = 294.0
rho = 1
FDT = (kB * T) / (rho)
FDT

In [ ]:
delta_nu_seed_avg = 0
for i in range(N_seeds):
    delta_nu_seed_avg += delta_nu_arr_chk_seeds[i][chk_idx][kmax_idx]

delta_nu_seed_avg = delta_nu_seed_avg / N_seeds

In [ ]:
delta_nu_seed_avg

In [ ]:
(delta_nu_seed_avg * kB * T) / (rho)